# 🌿 Plant Disease Classification System — Gradio Web Interface
**Deploy your trained model as an interactive web app directly inside Google Colab.**

| Step | Description |
|------|-------------|
| 1 | Setup & model loading |
| 2 | Class names & disease info dictionary |
| 3 | Prediction function |
| 4 | Gradio interface design |
| 5 | Launch with public URL |

## ⚙️ Section 1 — Setup & Installation

In [ ]:
# Install required libraries
!pip install -q gradio==4.44.1
!pip install -q tensorflow pillow numpy

In [ ]:
import numpy as np
import gradio as gr
import tensorflow as tf
from tensorflow import keras
from PIL import Image
import os

print(f'TensorFlow : {tf.__version__}')
print(f'Gradio     : {gr.__version__}')

In [ ]:
# ── Load saved model ─────────────────────────────────────────────────────────
MODEL_PATH = 'plant_disease_model.h5'   # change if saved elsewhere

print(f'Loading model from: {MODEL_PATH} ...')
model = keras.models.load_model(MODEL_PATH)
print('✅ Model loaded successfully!')
print(f'   Input shape  : {model.input_shape}')
print(f'   Output shape : {model.output_shape}')

## 📋 Section 2 — Class Names & Disease Info Dictionary

In [ ]:
# ── 38 PlantVillage class names (alphabetical — matches ImageDataGenerator order) ──
CLASS_NAMES = [
    'Apple___Apple_scab',
    'Apple___Black_rot',
    'Apple___Cedar_apple_rust',
    'Apple___healthy',
    'Blueberry___healthy',
    'Cherry_(including_sour)___Powdery_mildew',
    'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___healthy',
    'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot',
    'Peach___healthy',
    'Pepper,_bell___Bacterial_spot',
    'Pepper,_bell___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Raspberry___healthy',
    'Soybean___healthy',
    'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch',
    'Strawberry___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]

assert len(CLASS_NAMES) == 38, f'Expected 38 classes, got {len(CLASS_NAMES)}'
print(f'✅ {len(CLASS_NAMES)} class names defined.')

In [ ]:
# ── Disease information dictionary ────────────────────────────────────────────
# Contains detailed info for 15 key diseases + a generic fallback
DISEASE_INFO = {
    # ── HEALTHY (all crops) ──────────────────────────────────────────────────
    '_healthy': {
        'description': '✅ The plant appears HEALTHY with no visible disease signs.',
        'treatment'  : 'No treatment required. Continue standard care.',
        'prevention' : 'Maintain regular watering, fertilisation, and pest monitoring.',
        'severity'   : 'None'
    },
    # ── APPLE ────────────────────────────────────────────────────────────────
    'Apple___Apple_scab': {
        'description': '🍎 Apple Scab — fungal disease caused by Venturia inaequalis. '
                       'Causes olive-green to brown scab-like lesions on leaves and fruit.',
        'treatment'  : 'Apply fungicides (captan, myclobutanil, or sulfur) every 7–10 days '
                       'during wet spring weather. Remove and destroy infected leaves.',
        'prevention' : 'Plant resistant varieties; ensure good air circulation; '
                       'rake and dispose of fallen leaves in autumn.',
        'severity'   : 'Moderate'
    },
    'Apple___Black_rot': {
        'description': '🍎 Apple Black Rot — caused by Botryosphaeria obtusa. '
                       'Produces brown, rotting spots on fruit and cankers on branches.',
        'treatment'  : 'Prune infected wood; apply copper-based or captan fungicides. '
                       'Remove mummified fruit from trees.',
        'prevention' : 'Avoid wounding trees; ensure good drainage; inspect annually.',
        'severity'   : 'High'
    },
    'Apple___Cedar_apple_rust': {
        'description': '🍎 Cedar Apple Rust — caused by Gymnosporangium juniperi-virginianae. '
                       'Produces bright orange-yellow spots on apple leaves.',
        'treatment'  : 'Apply myclobutanil or mancozeb fungicides at pink bud stage. '
                       'Remove nearby cedar/juniper hosts if possible.',
        'prevention' : 'Plant rust-resistant apple varieties; create distance from junipers.',
        'severity'   : 'Moderate'
    },
    # ── TOMATO ───────────────────────────────────────────────────────────────
    'Tomato___Early_blight': {
        'description': '🍅 Tomato Early Blight — caused by Alternaria solani. '
                       'Shows dark brown concentric-ring lesions ("bull\'s-eye") on older leaves.',
        'treatment'  : 'Apply chlorothalonil or copper fungicide every 7 days. '
                       'Remove heavily infected leaves immediately.',
        'prevention' : 'Rotate crops every 2–3 years; mulch to prevent soil splash; '
                       'avoid overhead irrigation.',
        'severity'   : 'Moderate'
    },
    'Tomato___Late_blight': {
        'description': '🍅 Tomato Late Blight — caused by Phytophthora infestans. '
                       'Produces large, water-soaked, greasy-looking lesions on leaves and stems. '
                       'Highly destructive — the same pathogen caused the Irish Potato Famine.',
        'treatment'  : 'Apply mancozeb, chlorothalonil, or metalaxyl immediately. '
                       'Destroy infected plants — do NOT compost.',
        'prevention' : 'Use certified disease-free seed; avoid overhead watering; '
                       'scout fields regularly in cool, wet weather.',
        'severity'   : 'Critical'
    },
    'Tomato___Leaf_Mold': {
        'description': '🍅 Tomato Leaf Mold — caused by Passalora fulva (Fulvia fulva). '
                       'Pale green/yellow spots on upper leaf surface; olive-brown mold below.',
        'treatment'  : 'Improve ventilation; apply copper or chlorothalonil fungicide.',
        'prevention' : 'Keep humidity below 85%; prune lower leaves for air flow.',
        'severity'   : 'Moderate'
    },
    'Tomato___Septoria_leaf_spot': {
        'description': '🍅 Septoria Leaf Spot — caused by Septoria lycopersici. '
                       'Small circular spots with dark borders and lighter centers on lower leaves.',
        'treatment'  : 'Apply chlorothalonil or copper-based fungicide. Remove affected leaves.',
        'prevention' : 'Crop rotation; mulch; avoid wetting foliage when irrigating.',
        'severity'   : 'Moderate'
    },
    'Tomato___Bacterial_spot': {
        'description': '🍅 Bacterial Spot — caused by Xanthomonas spp. '
                       'Small, water-soaked spots that turn brown with yellow halos.',
        'treatment'  : 'Apply copper bactericide + mancozeb; avoid working with wet plants.',
        'prevention' : 'Use disease-free transplants; avoid overhead irrigation.',
        'severity'   : 'High'
    },
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': {
        'description': '🍅 Tomato Yellow Leaf Curl Virus (TYLCV) — transmitted by whiteflies. '
                       'Causes upward leaf curling, yellowing, stunted growth, and fruit loss.',
        'treatment'  : 'No cure once infected. Remove and destroy affected plants. '
                       'Control whitefly populations with imidacloprid or insecticidal soap.',
        'prevention' : 'Use TYLCV-resistant varieties; install insect-proof netting; '
                       'use reflective mulch to deter whiteflies.',
        'severity'   : 'Critical'
    },
    # ── POTATO ───────────────────────────────────────────────────────────────
    'Potato___Early_blight': {
        'description': '🥔 Potato Early Blight — caused by Alternaria solani. '
                       'Dark, concentric lesions on older leaves; reduces yield significantly.',
        'treatment'  : 'Apply chlorothalonil, mancozeb, or azoxystrobin every 7–10 days.',
        'prevention' : 'Crop rotation; use certified seed; maintain plant nutrition.',
        'severity'   : 'Moderate'
    },
    'Potato___Late_blight': {
        'description': '🥔 Potato Late Blight — caused by Phytophthora infestans. '
                       'Rapidly spreading water-soaked lesions; white mold on leaf underside.',
        'treatment'  : 'Apply metalaxyl + mancozeb immediately; hill soil around plants.',
        'prevention' : 'Plant resistant varieties; monitor forecasts for blight-risk conditions.',
        'severity'   : 'Critical'
    },
    # ── GRAPE ────────────────────────────────────────────────────────────────
    'Grape___Black_rot': {
        'description': '🍇 Grape Black Rot — caused by Guignardia bidwellii. '
                       'Brown leaf lesions with black pycnidia dots; fruit shrivels to black mummies.',
        'treatment'  : 'Apply mancozeb or myclobutanil from bud break; remove mummified fruit.',
        'prevention' : 'Prune for open canopy; destroy all mummies before budbreak.',
        'severity'   : 'High'
    },
    # ── CORN ─────────────────────────────────────────────────────────────────
    'Corn_(maize)___Common_rust_': {
        'description': '🌽 Corn Common Rust — caused by Puccinia sorghi. '
                       'Brick-red, oval pustules scattered on both leaf surfaces.',
        'treatment'  : 'Apply triazole or strobilurin fungicides at early rust detection.',
        'prevention' : 'Plant resistant hybrids; early planting to avoid peak spore periods.',
        'severity'   : 'Moderate'
    },
    'Corn_(maize)___Northern_Leaf_Blight': {
        'description': '🌽 Northern Corn Leaf Blight — caused by Exserohilum turcicum. '
                       'Long, cigar-shaped, grayish-green lesions on leaves.',
        'treatment'  : 'Apply propiconazole or azoxystrobin at early tasseling.',
        'prevention' : 'Resistant hybrids; crop rotation; tillage to reduce residue.',
        'severity'   : 'Moderate'
    },
    'Orange___Haunglongbing_(Citrus_greening)': {
        'description': '🍊 Huanglongbing (HLB / Citrus Greening) — caused by Candidatus Liberibacter, '
                       'spread by Asian citrus psyllid. Causes yellow mottling, lopsided bitter fruit, '
                       'and eventual tree death. Currently incurable.',
        'treatment'  : 'No cure. Remove and destroy infected trees to prevent spread. '
                       'Control psyllid with systemic insecticides.',
        'prevention' : 'Use certified disease-free nursery stock; install psyllid traps; '
                       'quarantine infected trees immediately.',
        'severity'   : 'Critical'
    },
}

# Generic fallback for diseases not in the dict
DISEASE_INFO['__default__'] = {
    'description': '⚠️ Disease detected. Consult a local agricultural extension officer for confirmation.',
    'treatment'  : 'Isolate affected plants. Apply broad-spectrum fungicide/bactericide as appropriate.',
    'prevention' : 'Practice crop rotation, maintain field hygiene, and use certified seeds.',
    'severity'   : 'Unknown'
}

SEVERITY_COLOR = {'None': '🟢', 'Moderate': '🟡', 'High': '🟠', 'Critical': '🔴', 'Unknown': '⚪'}

print(f'✅ Disease info dictionary ready ({len(DISEASE_INFO)-1} specific diseases + 1 fallback).')

## 🔮 Section 3 — Prediction Function

In [ ]:
IMG_SIZE = (224, 224)

def get_disease_info(class_name: str) -> dict:
    """Look up disease info; fall back to healthy or generic entry."""
    if class_name in DISEASE_INFO:
        return DISEASE_INFO[class_name]
    # Check if any part of the name matches a key
    for key in DISEASE_INFO:
        if key in class_name:
            return DISEASE_INFO[key]
    if '_healthy' in class_name.lower() or 'healthy' in class_name.lower():
        return DISEASE_INFO['_healthy']
    return DISEASE_INFO['__default__']


def format_class(raw: str) -> str:
    """Convert 'Tomato___Early_blight' → 'Tomato | Early Blight'."""
    parts = raw.split('___')
    crop  = parts[0].replace('_', ' ').strip()
    cond  = parts[1].replace('_', ' ').strip() if len(parts) > 1 else ''
    return f'{crop}  |  {cond}' if cond else crop


def predict_disease(image):
    """
    Main Gradio prediction function.

    Parameters
    ----------
    image : PIL.Image — uploaded image from Gradio

    Returns
    -------
    predicted_label : str
    confidence_str  : str
    top3_dict       : dict  (for gr.Label bar chart)
    info_md         : str   (markdown — disease description)
    treatment_md    : str   (markdown — treatment & prevention)
    """
    if image is None:
        return 'No image provided', '—', {}, '⚠️ Please upload an image.', ''

    # ── 1. Preprocess ────────────────────────────────────────────────────────
    img  = image.convert('RGB').resize(IMG_SIZE)   # PIL → RGB 224×224
    arr  = np.array(img, dtype=np.float32) / 255.0 # [0,1] normalisation
    inp  = np.expand_dims(arr, axis=0)              # (1, 224, 224, 3)

    # ── 2. Inference ─────────────────────────────────────────────────────────
    probs     = model.predict(inp, verbose=0)[0]    # (38,)
    top3_idx  = np.argsort(probs)[::-1][:3]

    best_idx  = top3_idx[0]
    best_name = CLASS_NAMES[best_idx]
    best_conf = float(probs[best_idx])

    # ── 3. Top-3 dict for Gradio Label component ─────────────────────────────
    top3_dict = {
        format_class(CLASS_NAMES[i]): round(float(probs[i]) * 100, 2)
        for i in top3_idx
    }

    # ── 4. Formatted outputs ─────────────────────────────────────────────────
    predicted_label = format_class(best_name)
    confidence_str  = f'{best_conf * 100:.2f}%'

    info = get_disease_info(best_name)
    sev_icon = SEVERITY_COLOR.get(info['severity'], '⚪')

    info_md = (
        f'### 🔬 Disease Details\n\n'
        f'**Severity:** {sev_icon} {info["severity"]}\n\n'
        f'{info["description"]}'
    )

    treatment_md = (
        f'### 💊 Treatment\n{info["treatment"]}\n\n'
        f'### 🛡️ Prevention\n{info["prevention"]}'
    )

    return predicted_label, confidence_str, top3_dict, info_md, treatment_md


print('✅ predict_disease() function ready.')

## 🎨 Section 4 — Gradio Interface Design

In [ ]:
# ── Collect example images from the dataset ───────────────────────────────────
DATASET_BASE = '/kaggle/input/plantvillage-dataset/plantvillage dataset/color'

example_images = []
if os.path.exists(DATASET_BASE):
    sample_classes = [
        'Tomato___Late_blight',
        'Tomato___Early_blight',
        'Apple___Apple_scab',
        'Potato___Late_blight',
        'Corn_(maize)___Common_rust_',
    ]
    for cls in sample_classes:
        cls_dir = os.path.join(DATASET_BASE, cls)
        if os.path.isdir(cls_dir):
            imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
            if imgs:
                example_images.append(os.path.join(cls_dir, imgs[0]))

print(f'Example images found: {len(example_images)}')

In [ ]:
# ── Build the Gradio interface ─────────────────────────────────────────────────
css = """
.gradio-container { font-family: 'Segoe UI', Arial, sans-serif; }
#title { text-align: center; color: #2d6a4f; }
#subtitle { text-align: center; color: #52796f; font-size: 1rem; margin-bottom: 10px; }
#pred-label { font-size: 1.4rem; font-weight: 700; color: #1b4332; }
#conf-label { font-size: 1.2rem; color: #40916c; }
.footer { text-align:center; color:#888; font-size:0.8rem; margin-top:10px; }
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue   = gr.themes.colors.green,
        secondary_hue = gr.themes.colors.emerald,
        font          = [gr.themes.GoogleFont('Inter'), 'Arial', 'sans-serif']
    ),
    css  = css,
    title= '🌿 Plant Disease Classification System'
) as demo:

    # ── Header ──────────────────────────────────────────────────────────────
    gr.Markdown('# 🌿 Plant Disease Classification System', elem_id='title')
    gr.Markdown(
        'Upload a leaf image to instantly detect plant disease using AI.  \n'
        'Powered by **EfficientNetB3** trained on the PlantVillage dataset (38 classes).',
        elem_id='subtitle'
    )

    with gr.Row():

        # ── Left column: Input ───────────────────────────────────────────────
        with gr.Column(scale=4):
            img_input = gr.Image(
                type          = 'pil',
                label         = '📷 Upload Leaf Image',
                sources       = ['upload', 'clipboard'],
                image_mode    = 'RGB',
                height        = 320,
                elem_id       = 'upload-box'
            )
            predict_btn = gr.Button(
                '🔍 Analyse Leaf', variant='primary', size='lg'
            )
            clear_btn   = gr.ClearButton(
                components=[img_input], value='🗑️ Clear', size='sm'
            )

        # ── Right column: Outputs ────────────────────────────────────────────
        with gr.Column(scale=6):

            with gr.Row():
                pred_label = gr.Textbox(
                    label='🌿 Detected Disease',
                    interactive=False,
                    elem_id='pred-label',
                    scale=3
                )
                conf_label = gr.Textbox(
                    label='📊 Confidence',
                    interactive=False,
                    elem_id='conf-label',
                    scale=1
                )

            top3_label = gr.Label(
                label='🏆 Top-3 Predictions',
                num_top_classes=3
            )

            with gr.Accordion('🔬 Disease Information', open=True):
                info_box = gr.Markdown(value='_Results will appear here after analysis._')

            with gr.Accordion('💊 Treatment & Prevention', open=False):
                treatment_box = gr.Markdown(value='')

    # ── Examples ────────────────────────────────────────────────────────────
    if example_images:
        gr.Examples(
            examples   = example_images,
            inputs     = img_input,
            label      = '📂 Try a Sample Image',
            examples_per_page = 5
        )

    # ── How-to guide ─────────────────────────────────────────────────────────
    with gr.Accordion('ℹ️ How to Use', open=False):
        gr.Markdown("""
        1. **Upload** a clear, close-up photo of a plant leaf (JPG / PNG)
        2. Click **Analyse Leaf** to run the AI model
        3. View the **detected disease**, **confidence score**, and **top-3 predictions**
        4. Expand **Disease Information** for symptoms and **Treatment & Prevention** for action steps

        > **Supported crops:** Apple, Blueberry, Cherry, Corn, Grape, Orange, Peach, Pepper, Potato, Raspberry, Soybean, Squash, Strawberry, Tomato
        """)

    gr.Markdown(
        '<p class="footer">🌿 Plant Disease Classification System &nbsp;|&nbsp; '
        'EfficientNetB3 + PlantVillage Dataset &nbsp;|&nbsp; Built with TensorFlow & Gradio</p>'
    )

    # ── Wire up the predict button ────────────────────────────────────────────
    predict_btn.click(
        fn      = predict_disease,
        inputs  = [img_input],
        outputs = [pred_label, conf_label, top3_label, info_box, treatment_box]
    )

    # Also trigger prediction on image upload for instant feedback
    img_input.upload(
        fn      = predict_disease,
        inputs  = [img_input],
        outputs = [pred_label, conf_label, top3_label, info_box, treatment_box]
    )

print('✅ Gradio interface built successfully.')

## 🚀 Section 5 — Launch the App

In [ ]:
# ── Launch Gradio app ─────────────────────────────────────────────────────────
# share=True  → generates a public gradio.live URL (valid for 72 hours)
# queue=True  → handles concurrent users gracefully
# debug=True  → prints full tracebacks in the Colab cell on errors

demo.queue(max_size=10).launch(
    share          = True,    # ← public URL
    debug          = True,    # ← verbose error output
    show_error     = True,
    server_name    = '0.0.0.0',
    server_port    = 7860,
    inbrowser      = False    # Colab doesn't support auto-open
)

In [ ]:
# ── (Optional) Close the app when done ───────────────────────────────────────
# Uncomment the line below to shut down the server
# demo.close()